# Grid Search hyper parameters for MAE-PU learning

1. Import dependencies

In [ ]:
import random
import torch
import numpy as np
import pandas as pd
import time
import glob
import os
import torch.utils.data as data

from src.util.torch import resolve_torch_device
from src.data.indian_pines import load_indian_pines
from src.definitions import GREED_SEARCH_FOLDER, RAW_DATA_FOLDER
from src.model.grid_search import GridSearch
from src.model.bin_mae_pu_grid_search import BinMaePuSearchAdapter
from src.util.hsi import pu_bin_train_test_split_by_mask, read_fixed_labels_mask
from src.visualization.plot import plot_masked_segmentation_comparison
from src.util.hsi import (
    extract_patches,
    preprocess_hsi,
    reduce_hsi_dim,
    DimReductionType,
    PreProcessType,
)
from src.util.dict_ext import unique_counts
from src.data.dataset_decorator import UnlabeledDatasetDecorator

2. Prepare env

In [ ]:
random_seed = 42

random.seed(random_seed)
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = resolve_torch_device()

generator = torch.Generator()
generator.manual_seed(random_seed)

In [ ]:
f"Device is {device}"

# Indian pines

0. Set params

In [ ]:
examples_per_class = 15
# epoch_seconds = int(time.time())
epoch_seconds = 1745249041
run_name = f"indian-pines-mae-pu-{epoch_seconds}"

In [ ]:
cpu_count = 4

f"Setting num_workers to {cpu_count}"

1. Load dataset

In [ ]:
target_class = 11
data_point_count = 245

In [ ]:
batch_size = 24
patch_size = 9
target_dim = 75

pre_process_type = PreProcessType.STANDARTIZATION
dim_reduction_type = DimReductionType.PCA

In [ ]:
image, labels = load_indian_pines()

In [ ]:
image_h, image_w, _ = image.shape

In [ ]:
_, image = preprocess_hsi(image, pre_process_type)

In [ ]:
_, target_dim, image = reduce_hsi_dim(
    image, target_dim, dim_reduction_type, device, random_seed
)

In [ ]:
x, y = extract_patches(image, labels, patch_size=patch_size)

In [ ]:
mask = read_fixed_labels_mask(
    f"indian-pines-train-test-split-{target_class}-{data_point_count}.npy",
    folder=RAW_DATA_FOLDER / "mask" / "train-test-split",
)

x_train, y_train, _, _ = pu_bin_train_test_split_by_mask(target_class, x, y, mask)


_ = plot_masked_segmentation_comparison(y.reshape(image_h, image_w), mask)

In [ ]:
y_train_unique = unique_counts(y_train)

y_train_unique

In [ ]:
y_target = y == target_class

In [ ]:
x_tensor = torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)
y_tensor = torch.tensor(y_target, dtype=torch.long)
x_train_tensor = torch.tensor(x_train, dtype=torch.float32).permute(0, 3, 1, 2)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

In [ ]:
train_dataset = data.TensorDataset(x_train_tensor, y_train_tensor)
full_dataset = data.TensorDataset(x_tensor, y_tensor)

train_loader = data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=cpu_count,
    persistent_workers=True,
)
full_loader = data.DataLoader(
    full_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=cpu_count,
    persistent_workers=True,
)
predict_loader = data.DataLoader(
    UnlabeledDatasetDecorator(full_dataset),
    batch_size=batch_size,
    num_workers=cpu_count,
    persistent_workers=True,
)

In [ ]:
positive_count = (y == target_class).sum()
all_count = len(y)
positive_prob = torch.tensor(positive_count / all_count, dtype=torch.float32)

In [ ]:
print(f"{target_class} count: {positive_count}")
print(f"Overall count: {all_count}")
print(f"Class prob: {positive_prob}")

2. Train model

In [ ]:
params = {
    "prediction_threshold": [0.5],
    "loss_fun": [lambda x: torch.nn.functional.softplus(-x)],
    "target_dim": [target_dim],
    "latent_space_size": [256],
    "patch_size": [9],
    "decoder_embed_dim": [256],
    "decoder_layers": [8],
    "decoder_heads": [16],
    "decoder_mlp_dim": [512],
    "learning_rate": [1e-3],
    "weight_decay": [0.0],
    "loss_alpha": [1.0],
    "loss_beta": [0.1],
    "autoencoder_loss_weight": [1.0],
    "cls_loss_weight": [1.0],
    "num_epochs": [120],
    "mask_ratio": [0.75],
    "mask_mode": ["spatial"],
    "fil_value": ["noise"],
}

adapter = BinMaePuSearchAdapter(
    params,
    train_dataloader=train_loader,
    eval_dataloader=full_loader,
    positive_prob=positive_prob,
    device=device,
)

In [ ]:
log_dir = GREED_SEARCH_FOLDER / run_name

In [ ]:
search = GridSearch(
    adapter=adapter,
    optimize_metric="eval_kappa",
    log_dir=log_dir,
    num_workers=cpu_count,
)

In [ ]:
_, best_params, best_score = search.run()

In [ ]:
print("Best Params:", best_params)
print("Best Score:", best_score)

3. Training results

In [ ]:
csv_files = glob.glob(os.path.join(log_dir, "*.csv"))

report = pd.concat([pd.read_csv(f) for f in csv_files])

report.head()

In [ ]:
len(report)

In [ ]:
report.sort_values("best_eval_kapp", ascending=False).head()